# Wells → Water System Spatial Join

Builds an inventory of municipal well permits inside the California Water Service
(Cal Water) **Los Altos Suburban** district, and renders an interactive map of the
wells colored by permit status.

**Pipeline:**
1. Download drinking-water system boundaries from the SWRCB FeatureServer
2. Load `WELLS.csv` — Valley Water well permits filtered to purpose *Municipal*
3. Point-in-polygon spatial join → `WELLS_with_water_system.csv`
4. Filter to `CWSC LOS ALTOS SUBURBAN` and map wells by STATUS → `los_altos_wells_map.html`

**Data sources:**
- [Valley Water well permits](https://data-valleywater.opendata.arcgis.com/datasets/43743ae1dbfa4172a3c4a0d6481093b7_0/explore)
- [SWRCB Drinking Water System Area Boundaries](https://gispublic.waterboards.ca.gov/portalserver/rest/services/Drinking_Water/California_Drinking_Water_System_Area_Boundaries/FeatureServer/0)


In [1]:
# pip install geopandas requests shapely pandas
# (skip any you already have)

import requests
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import json
import time

BASE_URL = (
    "https://gispublic.waterboards.ca.gov/portalserver/rest/services/"
    "Drinking_Water/California_Drinking_Water_System_Area_Boundaries/"
    "FeatureServer/0"
)
QUERY_URL = BASE_URL + "/query"

print("Dependencies loaded.")

Dependencies loaded.


In [2]:
# --- Step 1: Get total feature count ---

count_resp = requests.get(QUERY_URL, params={
    "where": "1=1",
    "returnCountOnly": "true",
    "f": "json"
})
count_resp.raise_for_status()
total = count_resp.json()["count"]
print(f"Total water system features: {total}")

Total water system features: 4988


In [3]:
# --- Step 2: Download all features (paginated at 1000/request) ---
# Requests WGS84 (4326) so the CRS matches the wells coordinates.

PAGE_SIZE = 1000
all_features = []

for offset in range(0, total, PAGE_SIZE):
    resp = requests.get(QUERY_URL, params={
        "where": "1=1",
        "outFields": "WATER_SYSTEM_NAME",
        "returnGeometry": "true",
        "outSR": "4326",
        "resultOffset": offset,
        "resultRecordCount": PAGE_SIZE,
        "f": "geojson"
    })
    resp.raise_for_status()
    features = resp.json().get("features", [])
    all_features.extend(features)
    print(f"  Fetched {len(all_features)} / {total}")
    time.sleep(0.2)  # be polite to the server

# Build a GeoDataFrame from the collected features
boundaries = gpd.GeoDataFrame.from_features(all_features, crs="EPSG:4326")
boundaries = boundaries[["WATER_SYSTEM_NAME", "geometry"]]
print(f"\nBoundaries GeoDataFrame: {len(boundaries)} rows, CRS={boundaries.crs}")
boundaries.head(3)

  Fetched 1000 / 4988
  Fetched 2000 / 4988
  Fetched 3000 / 4988
  Fetched 4000 / 4988
  Fetched 4988 / 4988

Boundaries GeoDataFrame: 4988 rows, CRS=EPSG:4326


,WATER_SYSTEM_NAME,geometry
0,ORLAND MOBILE H.P.,"POLYGON ((-122.19927 39.73471, -122.19927 39.7..."
1,BIG MEADOWS SUBDIVISION,"POLYGON ((-121.12489 40.19057, -121.12668 40.1..."
2,ANTOINETTE MUTUAL WATER CO,"POLYGON ((-122.20921 40.19319, -122.20918 40.1..."


In [4]:
# --- Step 3: Load wells CSV ---
# encoding='utf-8-sig' strips the BOM on the X column.

WELLS_PATH = "WELLS.csv"  # adjust path if needed

wells_df = pd.read_csv(WELLS_PATH, encoding="utf-8-sig")
print(f"Wells: {len(wells_df)} rows")
print(f"Columns: {wells_df.columns.tolist()}")
wells_df[["X", "Y"]].describe()

Wells: 2888 rows
Columns: ['X', 'Y', 'OBJECTID', 'STATUS', 'ORIG_PERMIT', 'CURRENT_PERMITNUM', 'CONSULTNUM', 'X_COORD', 'Y_COORD', 'BORE_DEPTH', 'BORE_DIAMETER', 'SCREEN_INTERVAL', 'WELL_TYPE_VALUE', 'WELL_PURPOSE_VALUE', 'GlobalID', 'CASING_DEPTH', 'CASING_DIAMETER', 'WELL_NBR']


,X,Y
count,2888.000000,2888.000000
mean,-121.911713,37.316806
std,0.157251,0.110628
min,-122.184727,36.945158
25%,-122.027802,37.280188
50%,-121.921712,37.356152
75%,-121.839741,37.396065
max,-121.379995,37.457992


In [5]:
# --- Step 4: Convert wells to GeoDataFrame ---

wells_gdf = gpd.GeoDataFrame(
    wells_df,
    geometry=gpd.points_from_xy(wells_df["X"], wells_df["Y"]),
    crs="EPSG:4326"
)
print(f"Wells GeoDataFrame ready. CRS={wells_gdf.crs}")

Wells GeoDataFrame ready. CRS=EPSG:4326


In [6]:
# --- Step 5: Spatial join (point in polygon) ---
# how='left' keeps all wells, including any that fall outside a boundary.

joined = gpd.sjoin(
    wells_gdf,
    boundaries,
    how="left",
    predicate="within"
)

# sjoin can produce duplicates if a point overlaps multiple polygons
# (e.g. at boundary edges). Keep the first match per well.
joined = joined[~joined.index.duplicated(keep="first")]

matched = joined["WATER_SYSTEM_NAME"].notna().sum()
unmatched = joined["WATER_SYSTEM_NAME"].isna().sum()
print(f"Matched:   {matched}")
print(f"Unmatched: {unmatched}  (wells outside any boundary polygon)")
joined[["X", "Y", "WELL_NBR", "WATER_SYSTEM_NAME"]].head(5)

Matched:   2704
Unmatched: 184  (wells outside any boundary polygon)


,X,Y,WELL_NBR,WATER_SYSTEM_NAME
0,-121.836743,37.314745,07S01E23D004,SAN JOSE WATER
1,-121.821806,37.300554,07S01E25D003,SAN JOSE WATER
2,-121.822115,37.300749,07S01E25D013,SAN JOSE WATER
3,-121.821794,37.300533,07S01E25D022,SAN JOSE WATER
4,-121.970534,37.259081,08S01W04R003,SAN JOSE WATER


In [7]:
# --- Step 6: Write output CSV ---
# Drops the geometry column and the sjoin index artifact.

OUTPUT_PATH = "WELLS_with_water_system.csv"

output_df = pd.DataFrame(joined.drop(columns=["geometry", "index_right"], errors="ignore"))
output_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(output_df)} rows to {OUTPUT_PATH}")

Saved 2888 rows to WELLS_with_water_system.csv


In [8]:
# --- Step 7: Map of CWSC LOS ALTOS SUBURBAN wells, colored by STATUS ---
# pip install folium   # (skip if you already have it)

import folium
from folium import FeatureGroup, LayerControl
from branca.element import MacroElement
from jinja2 import Template

TARGET_SYSTEM = "CWSC LOS ALTOS SUBURBAN"

# Use the in-memory result if present, otherwise reload the saved CSV
# so this cell also works after a fresh kernel restart.
try:
    source_df = output_df
except NameError:
    source_df = pd.read_csv("WELLS_with_water_system.csv")

map_df = source_df[source_df["WATER_SYSTEM_NAME"] == TARGET_SYSTEM].dropna(subset=["X", "Y"])
print(f"{TARGET_SYSTEM}: {len(map_df)} wells")

# Water-system boundary polygon. Reuse the in-memory `boundaries` GeoDataFrame
# if present, otherwise query the FeatureServer for just this one system.
try:
    target_boundary = boundaries[boundaries["WATER_SYSTEM_NAME"] == TARGET_SYSTEM]
except NameError:
    target_boundary = gpd.GeoDataFrame()

if len(target_boundary) == 0:
    _resp = requests.get(QUERY_URL, params={
        "where": f"WATER_SYSTEM_NAME = '{TARGET_SYSTEM}'",
        "outFields": "WATER_SYSTEM_NAME",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "geojson",
    })
    _resp.raise_for_status()
    target_boundary = gpd.GeoDataFrame.from_features(
        _resp.json().get("features", []), crs="EPSG:4326"
    )
print(f"Boundary polygons for system: {len(target_boundary)}")

# One color per STATUS value.
palette = [
    "red", "blue", "green", "purple", "orange",
    "darkred", "cadetblue", "darkgreen", "darkpurple", "black",
]
statuses = sorted(map_df["STATUS"].dropna().unique())
status_color = {s: palette[i % len(palette)] for i, s in enumerate(statuses)}

# Center the map on the mean well location.
m = folium.Map(
    location=[map_df["Y"].mean(), map_df["X"].mean()],
    zoom_start=13,
    tiles="OpenStreetMap",
)

# District boundary (added first so the well dots draw on top of it).
if len(target_boundary):
    folium.GeoJson(
        target_boundary,
        name="Water system boundary",
        style_function=lambda f: {
            "color": "#1f4e79",
            "weight": 3,
            "fill": True,
            "fillColor": "#1f4e79",
            "fillOpacity": 0.05,
        },
    ).add_to(m)
    # Zoom to the boundary extent: total_bounds = [minx, miny, maxx, maxy].
    minx, miny, maxx, maxy = target_boundary.total_bounds
    m.fit_bounds([[miny, minx], [maxy, maxx]])

# One toggleable FeatureGroup per STATUS, so the LayerControl can show/hide each.
groups = {}
for status in statuses:
    count = int((map_df["STATUS"] == status).sum())
    groups[status] = FeatureGroup(name=f"{status} ({count})").add_to(m)

for _, row in map_df.iterrows():
    color = status_color.get(row["STATUS"], "gray")
    folium.CircleMarker(
        location=[row["Y"], row["X"]],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        weight=1,
        popup=folium.Popup(
            f"<b>{row['WELL_NBR']}</b><br>Status: {row['STATUS']}",
            max_width=250,
        ),
        tooltip=str(row["STATUS"]),
    ).add_to(groups.get(row["STATUS"], m))

LayerControl(collapsed=False).add_to(m)


# Color legend as a MacroElement so it is injected into the map's own HTML
# (absolutely positioned within the map div -> never clipped by the iframe).
class StatusLegend(MacroElement):
    def __init__(self, color_map):
        super().__init__()
        self._name = "StatusLegend"
        self.color_map = color_map
        self._template = Template("""
        {% macro html(this, kwargs) %}
        <div style="position:absolute;bottom:30px;left:30px;z-index:9999;
                    background:white;padding:10px 12px;border:1px solid #999;
                    border-radius:6px;font-size:13px;line-height:1.6;
                    box-shadow:0 1px 4px rgba(0,0,0,0.3);">
          <b>STATUS</b>
          {% for status, color in this.color_map.items() %}
          <div><span style="display:inline-block;width:12px;height:12px;
               background:{{ color }};border-radius:50%;margin-right:6px;"></span>
               {{ status }}</div>
          {% endfor %}
        </div>
        {% endmacro %}
        """)

m.add_child(StatusLegend(status_color))

# VS Code's notebook webview often blocks external tile images, leaving the
# basemap blank inline. Saving to standalone HTML renders correctly in a browser.
MAP_PATH = "los_altos_wells_map.html"
m.save(MAP_PATH)
print(f"Saved map to {MAP_PATH} -- open it in a browser if the basemap is blank inline.")

m

CWSC LOS ALTOS SUBURBAN: 96 wells
Boundary polygons for system: 1
Saved map to los_altos_wells_map.html -- open it in a browser if the basemap is blank inline.
